# Week 8 — Impact Estimate

A rough, clearly-caveated back-of-envelope estimate of what the forecast-error reduction found in
`notebooks/21` (combining the from-scratch model with NDF beats NDF alone by 3.55%, verified on the
fresh, corrected 2025 `VALIDATION`) could plausibly be worth — in £ of avoided balancing cost and
tCO2 of avoided fossil-backup generation. **This is explicitly a directional estimate, not a causal
claim.** The arithmetic is shown in full below specifically so a reader can disagree with an
assumption rather than just the conclusion, per PLAN.md's own framing for this section.

**Why this magnitude, not an earlier one**: this project found several forecast-error reductions
along the way (e.g. weather features cutting MAE 27.65%, Week 3's tuning gains) against a naive
seasonal baseline that was never a real deployed alternative. The combination result is different —
it's the one improvement found *on top of* NDF, an already-deployed, already-accurate operational
forecast. It's the most honest answer to "what does this project's own work actually add," so it's
the one used here.

## Data sources

- **NESO — Daily Balancing Services Use of System (BSUoS) Cost Data**, FY2024-25 (2024-04-01 to
  2025-03-31), half-hourly, six cost categories (Energy Imbalance, Frequency Control, Positive
  Reserve, Constraints, Negative Reserve, Other). BSUoS recovers the day-to-day cost of operating
  the transmission system — the closest free, systematically-published proxy for "balancing cost"
  available. Saved to `data/raw/bsuos_costs_fy2024_25.csv`.
- **Carbon Intensity API** (`api.carbonintensity.org.uk`) — `/intensity/factors` gives standard
  gCO2/kWh factors per generation type; used here for Gas (Combined Cycle) and Gas (Open Cycle) as
  proxies for the marginal generation covering short-notice imbalance.
- **This project's own canonical demand table**, matched to the *exact same* FY2024-25 dates as the
  BSUoS data, to compute a same-period £/MWh rate rather than mismatching calendar years.

In [1]:
import pandas as pd

pd.options.display.float_format = "{:.4f}".format

bsuos = pd.read_csv("../data/raw/bsuos_costs_fy2024_25.csv")
cost_cols = ["Energy Imbalance", "Frequency Control", "Positive Reserve", "Constraints", "Negative Reserve", "Other"]
bsuos["total_cost_gbp"] = bsuos[cost_cols].sum(axis=1)

print(f"BSUoS data: {bsuos['SETT_DATE'].min()} to {bsuos['SETT_DATE'].max()}, n={len(bsuos)} settlement periods")
total_bsuos_gbp = bsuos["total_cost_gbp"].sum()
narrow_bsuos_gbp = bsuos[["Positive Reserve", "Energy Imbalance"]].sum().sum()
print(f"Total FY2024-25 BSUoS cost (all categories): £{total_bsuos_gbp:,.0f}")
print(f"Total FY2024-25 'Positive Reserve' + 'Energy Imbalance' only: £{narrow_bsuos_gbp:,.0f}")

BSUoS data: 2024-04-01 to 2025-03-31, n=17512 settlement periods
Total FY2024-25 BSUoS cost (all categories): £2,079,393,311
Total FY2024-25 'Positive Reserve' + 'Energy Imbalance' only: £93,759,716


## Two balancing-cost rates, deliberately — a broad one and a narrow one

**Broad rate**: all six BSUoS categories ÷ total demand. This treats the *entire* system-operation
cost as if it scaled with demand-forecast accuracy — a generous, almost certainly overstated
assumption, since `Constraints` (largely network congestion / wind curtailment payments) and
`Frequency Control` (second-by-second stability) are driven mostly by network topology and
generation-side factors, not day-ahead demand forecasting.

**Narrow rate**: only `Positive Reserve` + `Energy Imbalance` — the two categories most
mechanistically connected to "the system needed more generation/reserve than expected." Still an
imperfect proxy (captures *all* imbalance, including generation-forecast error and outages, not
demand-forecast error specifically), but a more defensible lower bound than the broad rate.

**Both are shown; the narrow one is the more credible headline number.**

In [2]:
df = pd.read_parquet("../data/processed/gb_energy_2020_2025.parquet")
demand_fy = df.loc["2024-04-01":"2025-03-31", "demand"]
total_demand_mwh = demand_fy.sum() * 0.5  # each row is a half-hour: MW * 0.5h = MWh

print(f"FY2024-25 GB demand: {len(demand_fy)} settlement periods, {total_demand_mwh / 1e6:.2f} TWh")

broad_rate_gbp_per_mwh = total_bsuos_gbp / total_demand_mwh
narrow_rate_gbp_per_mwh = narrow_bsuos_gbp / total_demand_mwh
print(f"\nBroad rate (all BSUoS / total demand): £{broad_rate_gbp_per_mwh:.4f} / MWh")
print(f"Narrow rate (reserve + imbalance / total demand): £{narrow_rate_gbp_per_mwh:.4f} / MWh")

FY2024-25 GB demand: 17520 settlement periods, 232.42 TWh

Broad rate (all BSUoS / total demand): £8.9466 / MWh
Narrow rate (reserve + imbalance / total demand): £0.4034 / MWh


## The avoided-error volume, from `notebooks/21`'s final result

MAE fell from 593.14 MW (NDF alone) to 572.11 MW (combined), evaluated on the full, corrected 2025
`VALIDATION` year. Treating that 21.03 MW average absolute-error reduction as if it applied to every
settlement period of a year gives an "avoided forecast-uncertainty volume" in MWh — the core
simplifying assumption of this whole estimate: that balancing cost scales roughly proportionally
with the *average magnitude* of demand-forecast error, at the same system-wide rate computed above.
Real balancing cost is driven by many more factors than demand-forecast error alone (generation
forecast error, plant outages, network constraints); this assumption is what makes the result
*directional*, not causal.

In [3]:
ndf_alone_mae = 593.14
combined_mae = 572.11
avoided_error_mw = ndf_alone_mae - combined_mae
avoided_error_mwh_per_year = avoided_error_mw * 8760  # sustained across a full year

print(f"Avoided average absolute forecast error: {avoided_error_mw:.2f} MW")
print(f"Equivalent avoided-uncertainty volume over a year: {avoided_error_mwh_per_year:,.0f} MWh")
print(f"As a share of GB's annual demand ({total_demand_mwh / 1e6:.1f} TWh): "
      f"{avoided_error_mwh_per_year / total_demand_mwh * 100:.3f}%")

Avoided average absolute forecast error: 21.03 MW
Equivalent avoided-uncertainty volume over a year: 184,223 MWh
As a share of GB's annual demand (232.4 TWh): 0.079%


## £ impact: applying both balancing-cost rates

In [4]:
impact_broad_gbp = avoided_error_mwh_per_year * broad_rate_gbp_per_mwh
impact_narrow_gbp = avoided_error_mwh_per_year * narrow_rate_gbp_per_mwh

print(f"Estimated annual value, broad rate:  £{impact_broad_gbp:,.0f}")
print(f"Estimated annual value, narrow rate: £{impact_narrow_gbp:,.0f}")
print(f"\nDirectional range: roughly £{impact_narrow_gbp:,.0f} to £{impact_broad_gbp:,.0f} per year")
print(f"For scale: FY2024-25 total BSUoS spend was £{total_bsuos_gbp:,.0f} -- "
      f"this range is {impact_narrow_gbp/total_bsuos_gbp*100:.3f}% to {impact_broad_gbp/total_bsuos_gbp*100:.3f}% of it")

Estimated annual value, broad rate:  £1,648,169
Estimated annual value, narrow rate: £74,316

Directional range: roughly £74,316 to £1,648,169 per year
For scale: FY2024-25 total BSUoS spend was £2,079,393,311 -- this range is 0.004% to 0.079% of it


## tCO2 impact: same avoided-volume, applied to marginal-gas carbon intensity

Uses the Carbon Intensity API's standard generation-type factors, as a proxy for "the marginal
generation most likely covering short-notice imbalance." **CCGT** (Combined Cycle Gas Turbine) is
GB's typical marginal/mid-merit generation; **OCGT** (Open Cycle Gas Turbine) is the faster-reacting,
more expensive peaking plant more specifically associated with short-notice balancing reserve — used
here as a plausible upper bound on carbon intensity, not a claim that OCGT covers all of it.

In [5]:
import requests

factors = requests.get("https://api.carbonintensity.org.uk/intensity/factors", timeout=30).json()["data"][0]
ccgt_g_per_kwh = factors["Gas (Combined Cycle)"]
ocgt_g_per_kwh = factors["Gas (Open Cycle)"]
print(f"CCGT: {ccgt_g_per_kwh} gCO2/kWh   OCGT: {ocgt_g_per_kwh} gCO2/kWh  (live, api.carbonintensity.org.uk)")

avoided_error_kwh_per_year = avoided_error_mwh_per_year * 1000
tco2_ccgt = avoided_error_kwh_per_year * ccgt_g_per_kwh / 1e6  # g -> tonnes
tco2_ocgt = avoided_error_kwh_per_year * ocgt_g_per_kwh / 1e6

print(f"\nEstimated annual avoided emissions, if covered by CCGT: {tco2_ccgt:,.0f} tCO2")
print(f"Estimated annual avoided emissions, if covered by OCGT: {tco2_ocgt:,.0f} tCO2")
print(f"Directional range: roughly {tco2_ccgt:,.0f} to {tco2_ocgt:,.0f} tCO2 per year")

CCGT: 394 gCO2/kWh   OCGT: 651 gCO2/kWh  (live, api.carbonintensity.org.uk)

Estimated annual avoided emissions, if covered by CCGT: 72,584 tCO2
Estimated annual avoided emissions, if covered by OCGT: 119,929 tCO2
Directional range: roughly 72,584 to 119,929 tCO2 per year


## Summary

| | low end (narrow £ rate / CCGT) | high end (broad £ rate / OCGT) |
|---|---|---|
| Avoided balancing cost, £/year | ~£74,000 | ~£1,650,000 |
| Avoided emissions, tCO2/year | ~72,600 | ~120,000 |

**Read this as an order-of-magnitude, directional sketch, not a forecast.** The wide range between
the two ends is the point, not a flaw to resolve — it reflects genuine uncertainty about how much of
GB's balancing-cost bill is actually sensitive to demand-forecast accuracy specifically, which this
back-of-envelope calculation cannot resolve with public aggregate data alone. The **narrow-rate,
CCGT-based low end (~£74k, ~72,600 tCO2 per year) is the more mechanistically defensible number** if
a single figure is wanted — it uses only the two BSUoS cost categories most directly tied to
imbalance, and GB's typical (not most extreme) marginal gas plant.

**What this number represents**: not "this project's model saves the system this much" — NDF is
NESO's own operational forecast, not something this project's model replaces. It's "combining a
free, from-scratch model with NESO's own forecast, a technique not currently known to be in use, is
worth roughly this much if it were adopted operationally" — a smaller, more honest claim than
"beating NESO," and arguably a more useful one for an EA-relevant "does this actually matter"
framing.